In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Literal
import os
from typing import TypedDict, Annotated
import operator
from operator import add
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

if not os.getenv('GOOGLE_API_KEY'):
    raise ValueError('Add GOOGLE_API_KEY to a .env file before running.')

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
);

In [4]:
class JokeState(TypedDict):
    topic : str
    joke: Annotated[list[str], add]
    explain: Annotated[list[str], add]


In [5]:
def generate_joke(state: JokeState):
    topic = state["topic"]
    prompt = f"Write a short, funny joke about {topic}"
    response = llm.invoke(prompt)
    return {"joke": [response.content]}

In [6]:
def generate_explanation(state: JokeState):
    joke = state["joke"][-1]
    prompt = f"Explain this joke simply in 1-2 sentences: {joke}"
    response = llm.invoke(prompt)
    return {"explain": [response.content]}

In [7]:

Graph = StateGraph(JokeState)

Graph.add_node("generate_joke", generate_joke)
Graph.add_node("generate_explanation", generate_explanation)

Graph.add_edge(START, "generate_joke")
Graph.add_edge("generate_joke", "generate_explanation")
Graph.add_edge("generate_explanation", END)

In [8]:

checkpointer = InMemorySaver()
bot = Graph.compile(checkpointer=checkpointer)

config1 = {"configurable": {"thread_id": "1"}}

res1 = bot.invoke({"topic": ["football"]}, config=config1)

print("Thread 1 - weather")
print("Joke:", res1["joke"][-1])
print("Explanation:", res1["explain"][-1])

config2 = {"configurable": {"thread_id": "2"}}

res2 = bot.invoke({"topic": ["government"]}, config=config2)

print("\nThread 2 - Python")
print("Joke:", res2["joke"][-1])
print("Explanation:", res2["explain"][-1])

res3 = bot.invoke({"topic": ["university"]}, config=config1)

print("\nThread 1 - After Second Run")
print("All Topics:", res3["topic"])
print("All Jokes:", res3["joke"])
print("All Explanations:", res3["explain"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Thread 1 - weather
Joke: [{'type': 'text', 'text': "Why are football players so good at laundry?\n\nBecause they're always great at tackling loads!", 'extras': {'signature': 'EmAKXgFpFH0TybgDCMrk9uCMvcQkpF0tcDtlbInwzZNL2e9nYj1HvtVwQTUsXsRGjGTO86r5ZsBcMi6JswHRfrmIPrKt8Sm77EPz9K3KBcbpkDaWXaCNtbgG9ds58dQEhGg='}}]
Explanation: [{'type': 'text', 'text': 'The joke plays on the double meaning of'}, {'type': 'text', 'text': ' the word "load": football players practice tackling opponents on the field, but the punchline refers to tackling piles of dirty laundry instead.', 'extras': {'signature': 'EmAKXgFpFH0T+vaazyKlgSTGZaBHiSK+JC4rXcZeBDWFfwFigySAQ1ewF/x7nCVCxgcEygnh0xJ5I1l4AR2NZ4of6yV43hJi3xE0XsMql5JP0NAWwEwX211Ht6a5xnn5rQw='}}]

Thread 2 - Python
Joke: [{'type': 'text', 'text': 'My favorite government program is the "Department of Redundancy Department." \n\nIt’s where they pay people to do things that people are already doing, so that those people can tell the other people that they\'re doin

In [9]:
snapshot1 = bot.get_state(config1)
snapshot2 = bot.get_state(config2)

print("Thread 1 Saved State:", snapshot1.values)
print("Thread 2 Saved State:", snapshot2.values)

Thread 1 Saved State: {'topic': ['university'], 'joke': [[{'type': 'text', 'text': "Why are football players so good at laundry?\n\nBecause they're always great at tackling loads!", 'extras': {'signature': 'EmAKXgFpFH0TybgDCMrk9uCMvcQkpF0tcDtlbInwzZNL2e9nYj1HvtVwQTUsXsRGjGTO86r5ZsBcMi6JswHRfrmIPrKt8Sm77EPz9K3KBcbpkDaWXaCNtbgG9ds58dQEhGg='}}], [{'type': 'text', 'text': 'University is great. You pay tens of thousands of dollars to learn that the answer to 90% of life\'s problems is "due in 2 hours" and "worth 10% of my final grade."', 'extras': {'signature': 'EmAKXgFpFH0TlmQatstYM8ElLw/jIzSFLLamyQjABRAa9MSXVDkS0ZBsSsZw+YHLSkjOsdMJv+zMK4DAg4N+QEE85FN1DpR/zo0hSdN7Z3MK8VqyedvzsnzosC0/x/nbKjE='}}]], 'explain': [[{'type': 'text', 'text': 'The joke plays on the double meaning of'}, {'type': 'text', 'text': ' the word "load": football players practice tackling opponents on the field, but the punchline refers to tackling piles of dirty laundry instead.', 'extras': {'signature': 'EmAKXgFpFH0T+vaa